In [122]:
import json
import re
import time
from collections import Counter

import numpy as np
import torch
from langdetect import detect
from SPARQLWrapper import JSON, SPARQLWrapper

# import faiss
from tenacity import before_sleep_log, retry, wait_random_exponential
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

In [123]:
ONTOLOGY_MAPPINGS_DIR = "./"

## Collecting relation names and constraints

### Collecting labels and data types of relations

In [124]:
PROP_2_LABEL = {}
PROP_2_DATA_TYPE = {}

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

# SPARQL query for properties with data types: Item, Quantity, Point in time
query = """
SELECT ?property ?propertyLabel ?typeLabel WHERE {
  ?property a wikibase:Property .
  ?property wikibase:propertyType ?type .
  
  FILTER(!exists{?property wdt:P31 wd:Q18644427})

  VALUES ?type { wikibase:WikibaseItem wikibase:Quantity wikibase:Time }
  
  BIND(
    IF(?type = wikibase:WikibaseItem, "Item",
      IF(?type = wikibase:Quantity, "Quantity",
        IF(?type = wikibase:Time, "Point in time", "Unknown")
      )
    ) AS ?typeLabel
  )
  
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

sparql.setQuery(query)
sparql.setReturnFormat(JSON)

try:
    results = sparql.query().convert()

    for result in results["results"]["bindings"]:
        prop = result["property"]["value"].split("/")[-1]
        label = result.get("propertyLabel", {}).get("value", "No label")
        data_type = result.get("typeLabel", {}).get("value", "Unknown")

        PROP_2_LABEL[prop] = label
        PROP_2_DATA_TYPE[prop] = data_type

except Exception as e:
    print(f"Error executing SPARQL query: {e}")

In [125]:
len(PROP_2_LABEL), len(PROP_2_DATA_TYPE)

(2473, 2473)

In [126]:
set(PROP_2_DATA_TYPE.values())

{'Item', 'Point in time', 'Quantity'}

In [127]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2data_type.json", "w") as f:
    json.dump(PROP_2_DATA_TYPE, f)

In [128]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2label.json", "w") as f:
    json.dump(PROP_2_LABEL, f)

### Collecting relation aliases

In [129]:
len(set(PROP_2_LABEL.keys())), len(set(PROP_2_LABEL.values()))

(2473, 2473)

In [130]:
@retry(wait=wait_random_exponential(multiplier=1, max=60))
def get_property_aliases(property_id):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    query = f"""
    SELECT ?alias WHERE {{
      wd:{property_id} skos:altLabel ?alias .
      FILTER (lang(?alias) = "en")
    }}
    """

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    aliases = [result["alias"]["value"] for result in results["results"]["bindings"]]
    return aliases


PROP2ALIASES = {}

for property_id in tqdm(PROP_2_LABEL.keys()):
    PROP2ALIASES[property_id] = get_property_aliases(property_id)

100%|██████████| 2473/2473 [08:05<00:00,  5.09it/s]


In [131]:
alias_set = set()
alias_list = []
for prop, aliases in PROP2ALIASES.items():
    alias_set.update(aliases)
    alias_list.extend(aliases)
print(len(alias_set), len(alias_list))

8553 9161


In [132]:
for prop in PROP_2_LABEL:
    print(PROP_2_LABEL[prop], PROP2ALIASES[prop])

official residence ['home', 'palace', 'domicile', 'executive mansion', 'executive residence', 'gubernatorial mansion', 'lives at', 'lives in', 'presidential mansion', 'presidential palace', 'resides at', 'state house']
record label ['label', 'labels']
production company ['film studio', 'studio', 'ballet company', 'broadcasting company', 'motion picture studio', 'movie studio', 'produced by (company)', 'producer (company)', 'production house', 'theater company', 'theatre company', 'theatrical troupe']
copyright license ['license', 'licence', 'content licence', 'content license', 'copyright licence', 'software license']
programmed in ['language', 'programming language', 'written in']
location ['based in', 'held by', 'is in', 'place held', 'exact location', 'place', 'position', 'village', 'from', 'in', 'near', 'region', 'city', 'located', 'neighborhood', 'town', 'neighbourhood', 'suburb', 'whereabouts', 'venue', 'locality', 'located in', 'locale']
subclass of ['rdfs:subClassOf', 'kind of'

In [133]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2label.json", "w") as f:
    json.dump(PROP_2_LABEL, f)

with open(ONTOLOGY_MAPPINGS_DIR + "prop2aliases.json", "w") as f:
    json.dump(PROP2ALIASES, f)

In [134]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2label.json", "r") as f:
    PROP_2_LABEL = json.load(f)

with open(ONTOLOGY_MAPPINGS_DIR + "prop2aliases.json", "r") as f:
    PROP2ALIASES = json.load(f)

### Collecting subject and value constraints of relations

In [135]:
from SPARQLWrapper import JSON, SPARQLWrapper


@retry(wait=wait_random_exponential(multiplier=1, max=60))
def get_constraints(property_id):
    """Retrieve value-type and subject-type constraints for a specified Wikidata property."""

    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    query = f"""
    SELECT ?constraintType ?entity ?entityLabel WHERE {{
      VALUES ?property {{ wd:{property_id} }}  

      ?property p:P2302 ?statement.  # Property constraints
      ?statement ps:P2302 ?constraintEntity.  # Constraint type

      VALUES ?constraintEntity {{ wd:Q21510865 wd:Q21503250 }}  # Value-type & Subject-type constraints

      ?statement pq:P2308 ?entity.  # The constrained entity type (allowed type)

      BIND(
        IF(?constraintEntity = wd:Q21510865, "Value-type constraint", "Subject type constraint")
        AS ?constraintType
      )
    }}
    """
    # SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    constraints = {"Value-type constraint": [], "Subject type constraint": []}
    for result in results["results"]["bindings"]:
        constraints[result["constraintType"]["value"]].append(
            result["entity"]["value"].split("/")[-1]
        )

    return constraints


# Example usage:
property_id = "P40"  # Replace with any Wikidata property ID
constraints = get_constraints(property_id)

print(constraints)

{'Value-type constraint': ['Q5', 'Q729', 'Q4886', 'Q95074', 'Q178885', 'Q207174', 'Q795052', 'Q2135501', 'Q4271324', 'Q13002315', 'Q16979650', 'Q21070568', 'Q21070598', 'Q21191150', 'Q24334299', 'Q64520857', 'Q75855169', 'Q115537581'], 'Subject type constraint': ['Q5', 'Q729', 'Q4886', 'Q95074', 'Q178885', 'Q207174', 'Q215627', 'Q219160', 'Q795052', 'Q2135501', 'Q3046146', 'Q4271324', 'Q13002315', 'Q16979650', 'Q21070568', 'Q21070598', 'Q24334299', 'Q75855169', 'Q115537581']}


In [136]:
constraint_dict = {}

for prop in tqdm(PROP_2_LABEL.keys()):
    constraint_dict[prop] = get_constraints(prop)
    time.sleep(0.1)
len(constraint_dict)

100%|██████████| 2473/2473 [15:21<00:00,  2.68it/s] 


2473

In [137]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2data_type.json", "r") as f:
    PROP_2_DATA_TYPE = json.load(f)

In [138]:
PROP_2_DATA_TYPE

{'P263': 'Item',
 'P264': 'Item',
 'P272': 'Item',
 'P275': 'Item',
 'P277': 'Item',
 'P276': 'Item',
 'P279': 'Item',
 'P282': 'Item',
 'P287': 'Item',
 'P286': 'Item',
 'P289': 'Item',
 'P291': 'Item',
 'P301': 'Item',
 'P306': 'Item',
 'P344': 'Item',
 'P355': 'Item',
 'P358': 'Item',
 'P361': 'Item',
 'P360': 'Item',
 'P364': 'Item',
 'P366': 'Item',
 'P369': 'Item',
 'P371': 'Item',
 'P375': 'Item',
 'P376': 'Item',
 'P397': 'Item',
 'P399': 'Item',
 'P398': 'Item',
 'P400': 'Item',
 'P403': 'Item',
 'P405': 'Item',
 'P404': 'Item',
 'P407': 'Item',
 'P406': 'Item',
 'P408': 'Item',
 'P411': 'Item',
 'P410': 'Item',
 'P413': 'Item',
 'P412': 'Item',
 'P415': 'Item',
 'P414': 'Item',
 'P417': 'Item',
 'P418': 'Item',
 'P421': 'Item',
 'P423': 'Item',
 'P425': 'Item',
 'P427': 'Item',
 'P437': 'Item',
 'P447': 'Item',
 'P449': 'Item',
 'P451': 'Item',
 'P450': 'Item',
 'P453': 'Item',
 'P452': 'Item',
 'P457': 'Item',
 'P459': 'Item',
 'P461': 'Item',
 'P460': 'Item',
 'P463': 'Item

In [139]:
constraint_dict["P2294"]

{'Value-type constraint': [], 'Subject type constraint': ['Q56061']}

In [140]:
wo_constraint = []
for prop in constraint_dict:
    if (
        len(constraint_dict[prop]["Value-type constraint"]) == 0
        and len(constraint_dict[prop]["Subject type constraint"]) == 0
    ):
        wo_constraint.append(prop)
len(wo_constraint)

587

In [141]:
quantity_props = []
time_props = []
other_props = []
for prop in wo_constraint:
    if PROP_2_DATA_TYPE[prop] == "Quantity":
        quantity_props.append(prop)
    elif PROP_2_DATA_TYPE[prop] == "Point in time":
        time_props.append(prop)
    else:
        other_props.append(prop)

        # print(PROP_2_LABEL[prop])
len(time_props), len(quantity_props), len(other_props)

(29, 299, 259)

In [142]:
sum((28, 295, 258))

581

In [143]:
for prop in constraint_dict:
    if PROP_2_DATA_TYPE[prop] == "Point in time":
        constraint_dict[prop]["Value-type constraint"].append("Q186408")

    elif PROP_2_DATA_TYPE[prop] == "Quantity":
        constraint_dict[prop]["Value-type constraint"].append("Q309314")

In [144]:
wo_constraint = []
for prop in constraint_dict:
    if (
        len(constraint_dict[prop]["Value-type constraint"]) == 0
        and len(constraint_dict[prop]["Subject type constraint"]) == 0
    ):
        wo_constraint.append(prop)
len(wo_constraint)

259

In [145]:
wo_constraint = []
for prop in constraint_dict:
    if len(constraint_dict[prop]["Value-type constraint"]) == 0:
        constraint_dict[prop]["Value-type constraint"] = ["ANY"]
    if len(constraint_dict[prop]["Subject type constraint"]) == 0:
        constraint_dict[prop]["Subject type constraint"] = ["ANY"]
len(wo_constraint)

0

In [146]:
wo_constraint = []
for prop in constraint_dict:
    if (
        len(constraint_dict[prop]["Value-type constraint"]) == 0
        and len(constraint_dict[prop]["Subject type constraint"]) == 0
    ):
        wo_constraint.append(prop)
len(wo_constraint)

0

In [147]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2constraints.json", "w") as f:
    json.dump(constraint_dict, f)

In [148]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2constraints.json", "r") as f:
    constraint_dict = json.load(f)

In [149]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2data_type.json", "r") as f:
    PROP_2_DATA_TYPE = json.load(f)

## Colecting entities' metadata

In [150]:
entities = set()
for prop, constraint in constraint_dict.items():
    for const_type in constraint:
        for entity in constraint[const_type]:
            entities.add(entity)
entities = list(entities)

In [151]:
len(entities)

3667

### Collecting entities' hierarchy of superclasses

In [152]:
@retry(wait=wait_random_exponential(multiplier=1, max=60))
def get_subclass_hierarchy(entity_id):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    # SPARQL query to get all subclasses (direct and indirect) of the given entity
    query = f"""
      SELECT DISTINCT ?subclass ?subclassLabel WHERE {{
          {{
              wd:{entity_id} wdt:P31/wdt:P279* ?subclass.
          }}
            UNION
          {{
              wd:{entity_id} wdt:P279* ?subclass.
          }}
      }}
      """
    # SERVICE wikibase:label {{ bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }}

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    results = sparql.query().convert()

    subclass_hierarchy = []

    for result in results["results"]["bindings"]:
        subclass_id = result["subclass"]["value"].split("/")[-1]
        subclass_hierarchy.append(subclass_id)

    return subclass_hierarchy


ENTITY_2_HIERARCHY = {}
for entity_id in tqdm(entities):
    hierarchy = get_subclass_hierarchy(entity_id)
    ENTITY_2_HIERARCHY[entity_id] = hierarchy

len(ENTITY_2_HIERARCHY)

100%|██████████| 3667/3667 [20:46<00:00,  2.94it/s]


3667

In [153]:
ents = []
for item in ENTITY_2_HIERARCHY.values():
    ents.extend(item)
len(set(ents)), len(entities)

(7578, 3667)

In [154]:
# leaving only entity types that are used in constraints
for entity in tqdm(ENTITY_2_HIERARCHY):
    filtered_super_entities = [
        item for item in ENTITY_2_HIERARCHY[entity] if item in entities
    ]
    ENTITY_2_HIERARCHY[entity] = filtered_super_entities

100%|██████████| 3667/3667 [00:02<00:00, 1325.08it/s]


In [155]:
ents = []
for item in ENTITY_2_HIERARCHY.values():
    ents.extend(item)
len(set(ents)), len(entities)

(3667, 3667)

In [156]:
with open(ONTOLOGY_MAPPINGS_DIR + "entity_hierarchy.json", "w") as f:
    json.dump(ENTITY_2_HIERARCHY, f)

### Collecting entity types' labels

In [157]:
BATCH_SIZE = 50


@retry(wait=wait_random_exponential(multiplier=1, max=60))
def fetch_labels(batch):
    entity_values = " ".join(f"wd:{entity}" for entity in batch)

    query = f"""
    SELECT ?entity ?entityLabel WHERE {{
      VALUES ?entity {{ {entity_values} }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    try:
        results = sparql.query().convert()
        return {
            result["entity"]["value"].split("/")[-1]: result.get("entityLabel", {}).get(
                "value", "No label"
            )
            for result in results["results"]["bindings"]
        }
    except Exception as e:
        print(f"Error with batch {batch[:5]}...: {e}")
        return {}


ENTITY_2_LABEL = {}

for i in range(0, len(entities), BATCH_SIZE):
    batch = entities[i : i + BATCH_SIZE]
    print(f"Processing batch {i // BATCH_SIZE + 1}/{(len(entities) // BATCH_SIZE) + 1}")

    labels = fetch_labels(batch)
    ENTITY_2_LABEL.update(labels)

# for entity, label in all_labels.items():
#     print(f"{entity}: {label}")

Processing batch 1/74
Processing batch 2/74
Processing batch 3/74
Processing batch 4/74
Processing batch 5/74
Processing batch 6/74
Processing batch 7/74
Processing batch 8/74
Processing batch 9/74
Processing batch 10/74
Processing batch 11/74
Processing batch 12/74
Processing batch 13/74
Processing batch 14/74
Processing batch 15/74
Processing batch 16/74
Processing batch 17/74
Processing batch 18/74
Processing batch 19/74
Processing batch 20/74
Processing batch 21/74
Processing batch 22/74
Processing batch 23/74
Processing batch 24/74
Processing batch 25/74
Processing batch 26/74
Processing batch 27/74
Processing batch 28/74
Processing batch 29/74
Processing batch 30/74
Processing batch 31/74
Processing batch 32/74
Processing batch 33/74
Processing batch 34/74
Processing batch 35/74
Processing batch 36/74
Processing batch 37/74
Processing batch 38/74
Processing batch 39/74
Processing batch 40/74
Processing batch 41/74
Processing batch 42/74
Processing batch 43/74
Processing batch 44/

In [158]:
len(ENTITY_2_LABEL)

3667

In [159]:
len(set(ENTITY_2_LABEL.keys())), len(set(ENTITY_2_LABEL.values()))

(3667, 3609)

In [160]:
label2entity = {}
for entity, label in ENTITY_2_LABEL.items():
    if label not in label2entity:
        label2entity[label] = []
    label2entity[label].append(entity)

for label, entities in label2entity.items():
    if len(entities) > 1:
        print(label, entities)

order ['Q567696', 'Q193622']
competition ['Q476300', 'Q841654']
role ['Q214339', 'Q4897819', 'Q1707847']
cabinet ['Q6866562', 'Q640506']
statement ['Q2684591', 'Q613299']
legal act ['Q740464', 'Q1864008']
dissertation ['Q1385450', 'Q3030630']
article ['Q191067', 'Q712597']
theatre company ['Q11812394', 'Q742421']
person ['Q215627', 'Q690940']
epithet ['Q16685255', 'Q207869']
group ['Q83478', 'Q16887380']
space ['Q2133296', 'Q107']
work ['Q386724', 'Q268378']
color ['Q1075', 'Q22006653']
process ['Q10843872', 'Q3249551']
attribute ['Q109674924', 'Q2722260']
achievement ['Q2532754', 'Q2988681']
test ['Q1003030', 'Q27318']
crossing ['Q10816681', 'Q62059481']
commission ['Q55657615', 'Q63705303']
motif ['Q68614425', 'Q1229071']
operator ['Q29933786', 'Q131030']
lineage ['Q1517820', 'Q1642895']
clan ['Q211503', 'Q989470']
parish ['Q102496', 'Q7137411']
style ['Q1292119', 'Q2313235', 'Q5767753']
language ['Q315', 'Q4113741', 'Q34770']
design ['Q82604', 'Q12826253']
position ['Q4164871', 'Q17

### Collecting descriptions for entity types with duplicated labels

In [161]:
@retry(wait=wait_random_exponential(multiplier=1, max=60))
def get_entity_info(entity_id):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    query = f"""
    SELECT ?entityLabel ?entityDescription WHERE {{
      wd:{entity_id} rdfs:label ?entityLabel .
      wd:{entity_id} schema:description ?entityDescription .
      FILTER (lang(?entityLabel) = "en")
      FILTER (lang(?entityDescription) = "en")
    }}
    """

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    if results["results"]["bindings"]:
        result = results["results"]["bindings"][0]
        return {
            "label": result["entityLabel"]["value"],
            "description": result["entityDescription"]["value"],
        }
    else:
        return None


for label, entities in label2entity.items():
    if len(entities) > 1:
        for entity_id in entities:
            info = get_entity_info(entity_id)
            ENTITY_2_LABEL[entity_id] = info["label"] + " (" + info["description"] + ")"

In [162]:
len(set(ENTITY_2_LABEL.keys())), len(set(ENTITY_2_LABEL.values()))

(3667, 3667)

### Collecting entity types' aliases

In [163]:
@retry(wait=wait_random_exponential(multiplier=1, max=60))
def get_entity_aliases(entity_id):
    chinese_japanese_pattern = re.compile(
        r"[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF\u3040-\u309F\u30A0-\u30FF\u31F0-\u31FF\uFF00-\uFFEF]"
    )
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")

    query = f"""
    SELECT ?alias WHERE {{
      wd:{entity_id} skos:altLabel ?alias .
      FILTER (lang(?alias) = "en")
    }}
    """

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    aliases = []
    for result in results["results"]["bindings"]:
        alias = result["alias"]["value"]
        if not chinese_japanese_pattern.search(alias):
            aliases.append(alias)
        # except Exception as e:
        #    continue

    return aliases


ENTITY_2_ALIASES = {}
for entity_id in tqdm(ENTITY_2_LABEL.keys()):
    ENTITY_2_ALIASES[entity_id] = get_entity_aliases(entity_id)

100%|██████████| 3667/3667 [13:50<00:00,  4.42it/s]


In [164]:
# [(ENTITY_2_LABEL[ent], aliases) for ent, aliases in ENTITY_2_ALIASES.items()]

In [165]:
ENTITY_2_LABEL["Q186408"], get_entity_aliases("Q186408")

('point in time',
 ['moment', 'date', 'instant', 'given moment', 'time point', 'timepoint'])

In [166]:
ENTITY_2_LABEL["Q309314"], get_entity_aliases("Q309314")

('quantity', ['number', 'amount', 'qty', 'quantulum'])

## Building inverse mapping - object/subject type constraint to relation

### Checking that relations with 'point in time' and 'quantity' data types don't have other constraints

In [167]:
# for prop, data_type in PROP_2_DATA_TYPE.items():
#     if data_type == "Quantity":
#         val_constraints = [ENTITY_2_LABEL[ent] for ent in constraint_dict[prop]["Value-type constraint"]]
#         subj_constraints = [ENTITY_2_LABEL[ent] for ent in constraint_dict[prop]["Subject type constraint"]]
#         # print(PROP_2_LABEL[prop], subj_constraints, val_constraints)
#         assert len(val_constraints) == 0
#         # no value constraints for data type quantity

In [168]:
# for prop, data_type in PROP_2_DATA_TYPE.items():
#     if data_type == "Point in time":
#         val_constraints = [ENTITY_2_LABEL[ent] for ent in constraint_dict[prop]["Value-type constraint"]]
#         subj_constraints = [ENTITY_2_LABEL[ent] for ent in constraint_dict[prop]["Subject type constraint"]]
#         # print(PROP_2_LABEL[prop], subj_constraints, val_constraints)
#         assert len(val_constraints) == 0
#         # no value constraints for data type quantity

In [169]:
constraint_dict

{'P263': {'Value-type constraint': ['Q41176', 'Q481289', 'Q15720793'],
  'Subject type constraint': ['Q4164871']},
 'P264': {'Value-type constraint': ['Q18127',
   'Q210167',
   'Q1137109',
   'Q1542343'],
  'Subject type constraint': ['Q5',
   'Q7889',
   'Q106833',
   'Q134556',
   'Q193977',
   'Q215380',
   'Q234870',
   'Q482994',
   'Q2088357',
   'Q3331189',
   'Q7725310',
   'Q11664239',
   'Q55155641',
   'Q105815710',
   'Q108346082',
   'Q115669410']},
 'P272': {'Value-type constraint': ['Q43229', 'Q1298801', 'Q11396960'],
  'Subject type constraint': ['Q386724']},
 'P275': {'Value-type constraint': ['Q77205602'],
  'Subject type constraint': ['Q8513',
   'Q42848',
   'Q49848',
   'Q386724',
   'Q1574516',
   'Q2668072',
   'Q3052382',
   'Q14897293',
   'Q16889133',
   'Q18593264']},
 'P277': {'Value-type constraint': ['Q9143', 'Q165194', 'Q188860'],
  'Subject type constraint': ['Q7397', 'Q8513', 'Q124590', 'Q3428776']},
 'P276': {'Value-type constraint': ['Q4130',
   'Q69

In [170]:
subj2prop_constraints = {"<ANY SUBJECT>": []}
# Q309314 - quantity, Q186408 -  point in time
obj2prop_constraint = {"<ANY OBJECT>": [], "Q309314": [], "Q186408": []}

for prop, constraint in constraint_dict.items():
    if PROP_2_DATA_TYPE[prop] == "Point in time":
        obj2prop_constraint["Q186408"].append(prop)

    elif PROP_2_DATA_TYPE[prop] == "Quantity":
        obj2prop_constraint["Q309314"].append(prop)

    elif constraint["Value-type constraint"] == ["ANY"]:
        obj2prop_constraint["<ANY OBJECT>"].append(prop)

    else:
        for entity in constraint["Value-type constraint"]:
            if entity not in obj2prop_constraint:
                obj2prop_constraint[entity] = []
            obj2prop_constraint[entity].append(prop)

    if constraint["Subject type constraint"] == ["ANY"]:
        subj2prop_constraints["<ANY SUBJECT>"].append(prop)

    else:
        for entity in constraint["Subject type constraint"]:
            if entity not in subj2prop_constraints:
                subj2prop_constraints[entity] = []
            subj2prop_constraints[entity].append(prop)


len(subj2prop_constraints), len(obj2prop_constraint)

(2352, 2250)

In [171]:
with open(ONTOLOGY_MAPPINGS_DIR + "subj_constraint2prop.json", "w") as f:
    json.dump(subj2prop_constraints, f)

with open(ONTOLOGY_MAPPINGS_DIR + "obj_constraint2prop.json", "w") as f:
    json.dump(obj2prop_constraint, f)

In [172]:
with open(ONTOLOGY_MAPPINGS_DIR + "entity_type2label.json", "w") as f:
    json.dump(ENTITY_2_LABEL, f)

In [173]:
with open(ONTOLOGY_MAPPINGS_DIR + "prop2aliases.json", "w") as f:
    json.dump(PROP2ALIASES, f)

In [174]:
with open(ONTOLOGY_MAPPINGS_DIR + "entity_type2aliases.json", "w") as f:
    json.dump(ENTITY_2_ALIASES, f)

In [175]:
with open(ONTOLOGY_MAPPINGS_DIR + "entity_type2hierarchy.json", "w") as f:
    json.dump(ENTITY_2_HIERARCHY, f)